# 🎬 The Walking Skeleton — the whole play on cardboard props

Chapters 02–04 gave you the *nouns*: validated shapes, the two ports, and the canonical
Ada/Bell story. This chapter makes them **move**. You will run the entire lifecycle —
need → quote → decide → pay → mint → activate → provision → tear down — inside this one
Python process, with every hard part replaced by a deliberately dumb stand-in.

**What you'll be able to do after this notebook**

- Explain what a *walking skeleton* is, and why the fakes being tiny is a feature, not a shortcut
- Run every deny path of `fulfill()` and *prove* a rejected payment leaves no trace (hand-made atomicity)
- Walk the authorization predicate line by line and trigger each `ErrorCode` on purpose
- Drive a session through both teardown paths: the clock (`tick`) and a revocation event
- Name, for every cardboard prop, the real component (and the chapter) that replaces it

**You need:** notebooks 00–04. No blockchain, no router, no LLM — this whole chapter is
plain in-process Python.

**Estimated time:** 60–90 minutes.

> This is the slow, fully decomposed tour. The compact one-pass version lives at
> [`e2e/notebooks/skeleton_explore.ipynb`](../skeleton_explore.ipynb) — worth a replay
> *after* you finish here.

## 1. Prove the joints, not the muscle

Picture building this system the obvious way: a month perfecting the smart contract, a
month on router automation, a month on the agents — and only *then* wiring them
together. The first time the pieces meet is the day everything is supposedly done, and
that is the day you learn the contract returns a ticket id the controller never
expected. An integration bug found that day is a catastrophe; the same bug found today
costs ten minutes.

The **walking skeleton** is the cure: perform the *entire* lifecycle on day one, end to
end, with every hard organ replaced by a **fake** — a cheap stand-in that does maybe 1%
of the real thing's job. The skeleton proves the **joints** (do the pieces fit and
talk?), not the **muscle** (is each piece strong?). Muscle arrives later, one organ at
a time, and after each transplant the same play must still run green.

First, the whole cast in one import cell — three groups: the ports and shared enums
(from 02/03), the canonical fixtures (from 04), and this chapter's stars, the skeleton
itself.

In [ ]:
import inspect
import time
from datetime import datetime, timezone

import pydantic

# the two ports (03) and the shared enums (02/04)
from a2a_interfaces import EntitlementReader, ErrorCode, NetworkProvisioner, SessionState

# the one canonical example — Ada / Bell / ticket #7 / 10 TOK / 50 Mbps (04)
from a2a_interfaces import fixtures as fx

# the cardboard props and the stub controller — this chapter's stars
from e2e.skeleton.fakes import (
    FakeChain,
    FakeClock,
    FakeNet,
    InsufficientFunds,
    OfferAlreadyUsed,
    OfferExpired,
    WrongConsumer,
)
from e2e.skeleton.scripted_agents import ScriptedConsumer, ScriptedProvider
from e2e.skeleton.stub_controller import Denied, StubController, predicate

print("cast imported: 2 ports, the fixtures (fx), 3 fakes, 2 scripted agents, 1 stub controller")

A fair worry: "if the blockchain is fake, what does running this prove?" Answer: the
fakes are kept so small they *can't lie*. A fake with real behavior — retries, its own
validation, clever caching — is a **second implementation** of the component, and two
implementations can disagree; then your tests pass against the lie. The project's plan
(docs/01, M0.3) says it plainly: *if a fake starts to look like a real Ethereum or a
real router, it is a bug.* Let's measure the whole "blockchain":

In [ ]:
import e2e.skeleton.fakes as fakes_mod

file_lines = len(inspect.getsource(fakes_mod).splitlines())
chain_lines = len(inspect.getsource(FakeChain).splitlines())
clock_lines = len(inspect.getsource(FakeClock).splitlines())
print("fakes.py, the whole file :", file_lines, "lines  ← clock + chain + net alias + 4 exceptions")
print("class FakeChain          :", chain_lines, "lines  ← the entire 'blockchain', comments included")
print("class FakeClock          :", clock_lines, "lines")
assert chain_lines < 100  # more behavior than this would be a second implementation
print("✓ small enough that it cannot lie")

**✏️ Your turn 1.1 — Predict the size of the router prop**

`FakeChain` measured under 100 lines. The skeleton's "router", `FakeNet`, lives by the same law — or does it? Write your prediction as a comment (over or under 100 lines? bigger or smaller than `FakeChain`?), then run the measurement. Success: a printed line count that settles your bet, and the un-commented self-check passing.

In [ ]:
# my prediction: FakeNet is ___ lines (over/under 100? bigger/smaller than FakeChain?)  # TODO: bet first

net_lines = len(inspect.getsource(FakeNet).splitlines())
print("class FakeNet, the whole 'router' :", net_lines, "lines")
print("class FakeChain, for comparison   :", chain_lines, "lines")
# TODO: un-comment the self-check once your bet is written down
# assert net_lines < 100  # a router fake with more behavior could lie too

<details><summary>✅ Solution 1.1 — peek only after trying</summary>

```python
net_lines = len(inspect.getsource(FakeNet).splitlines())
assert net_lines < 100
```

`FakeNet` is ~45 lines — a dict, a list, and four recording methods; small enough that it *can't* disagree with the real provisioner it stands in for (rule 7's contract tests check the rest).

</details>

## 2. FakeClock — time you can fast-forward

Everything Ada bought is time-boxed: the offer dies at 14:20, the service window runs
14:00 → 16:00 (you pinned those unix timestamps in 04). Which clock decides? **ADR-004:
chain time is the only clock.** Validity is judged by the blockchain's own timestamp,
never by the controller's OS clock. OS timers may *schedule wake-ups*, but every action
re-checks `chain_time()` before acting — two machines' wall clocks can disagree; there
is only one chain, so there is only one time.

For the skeleton, that clock is `FakeClock`: one integer you can shove forward. This is
the whole class:

In [ ]:
print(inspect.getsource(FakeClock))

Why a class around one integer? Because several parts of the system ask for the time at
different moments, and each must see the *current* value — one shared object, one
truth. Poke it, and notice what does **not** move it: `time.sleep()`. Real seconds
mean nothing here; only `advance()` (a new block's timestamp, in real life) counts.

In [ ]:
STORY_TIME = fx.WINDOW.start - 1680          # 13:32 — the story opens 28 min before Ada's window


def hhmm(ts: int) -> str:
    """Unix seconds -> the story's HH:MM (UTC), for readable narration."""
    return datetime.fromtimestamp(ts, tz=timezone.utc).strftime("%H:%M")


clk = FakeClock(STORY_TIME)
print("chain time:", clk.now(), f"= {hhmm(clk.now())}")

time.sleep(0.3)                              # real time passes...
assert clk.now() == STORY_TIME               # ...chain time does not care
print("after time.sleep(0.3) :", hhmm(clk.now()), " ← unmoved (ADR-004)")

clk.advance(60)                              # a 'block' lands a minute later
assert clk.now() == STORY_TIME + 60
print("after advance(60)     :", hhmm(clk.now()), " ← only advance() moves it")
assert hhmm(fx.WINDOW.start) == "14:00" and hhmm(fx.WINDOW.end) == "16:00"

**✏️ Your turn 2.1 — Land your own clock on curtain-up**

Build a fresh `FakeClock` at `STORY_TIME`, compute the exact number of seconds to the window start, and `advance()` it there in one move. Success: your clock prints `14:00` and the self-check passes.

In [ ]:
my_clk = FakeClock(STORY_TIME)

gap = 0    # TODO: compute the seconds separating my_clk from fx.WINDOW.start
my_clk.advance(gap)
print("my clock reads:", hhmm(my_clk.now()), " ← aim for 14:00")
# TODO: un-comment once your clock lands exactly on the window start
# assert my_clk.now() == fx.WINDOW.start and hhmm(my_clk.now()) == "14:00"

<details><summary>✅ Solution 2.1 — peek only after trying</summary>

```python
my_clk = FakeClock(STORY_TIME)
gap = fx.WINDOW.start - my_clk.now()     # 1680 s = the 28 minutes from the comment above
my_clk.advance(gap)
assert my_clk.now() == fx.WINDOW.start and hhmm(my_clk.now()) == "14:00"
```

Only `advance()` moves chain time (ADR-004), so "waiting 28 minutes" is a single arithmetic step — no sleeping, no wall clock.

</details>

## 3. FakeChain — a blockchain that is secretly three containers

The real chain does payments, NFT minting, signature checks, gas. `FakeChain` keeps
only the **bookkeeping** the story needs, in containers you already know:

- `balances` — dict of `address → TOK` (integer money units, from 04): who has what
- `consumed` — a *set* of offer salts already spent: the punched-serial ledger behind single-use
- `_entitlements` / `_owners` — dicts of `ticket id → EntitlementView` / `→ owner`: the ticket registry
- `_watchers` — a list of callbacks to ring when a ticket is revoked (they fire in §10)

Here is `__init__`, straight from the source:

In [ ]:
print(inspect.getsource(FakeChain.__init__))

Build one. We seed the stage exactly as the story wants it: Ada holds 50 TOK, Bell 0,
and the next ticket number is 7 (on the real chain Bell will have sold six tickets
before ours — chapter 06 shows that seeding). Names with a leading `_` are internal —
we peek only to learn.

In [ ]:
SEED = 50 * 10**18                 # Ada's 50 TOK in base units (04: integer money)

clock = FakeClock(STORY_TIME)
chain = FakeChain(clock, balances={fx.ADA: SEED, fx.BELL: 0}, next_id=fx.TICKET_ID)

print("balances     :", {a[:8] + "…": v // 10**18 for a, v in chain.balances.items()}, "(TOK)")
print("consumed     :", chain.consumed, " ← no salts punched yet")
print("_owners      :", chain._owners, " ← no tickets minted yet")
print("_entitlements:", chain._entitlements)
assert chain.balances[fx.ADA] == SEED and not chain.consumed and not chain._owners

One more thing before we spend money. In 03 you met the two ports. `FakeChain` never
*declares* itself an `EntitlementReader` — it simply *has* the four methods, and that
structural fit is the entire swap trick: **rule 7, a mock implements the SAME Protocol
as the real adapter**. Same story for `FakeNet`, the router-shaped recorder we employ
in §8. Check both fits side by side:

In [ ]:
net = FakeNet()
print("FakeChain satisfies EntitlementReader ?", isinstance(chain, EntitlementReader))
print("FakeNet   satisfies NetworkProvisioner?", isinstance(net, NetworkProvisioner))
assert isinstance(chain, EntitlementReader)
assert isinstance(net, NetworkProvisioner)
print("EntitlementReader wants:", [m for m in dir(EntitlementReader) if not m.startswith("_")])

**✏️ Your turn 3.1 — Predict the cross-fit**

Rule 7 says each fake fits *its own* port. Before running: does `chain` also satisfy `NetworkProvisioner`? Does `net` satisfy `EntitlementReader`? Write both predictions as comments, run, then un-comment the self-check. Success: two `False`s — and you can name a missing method behind each one.

In [ ]:
# chain satisfies NetworkProvisioner? my prediction: ___   # TODO: predict BEFORE running
# net   satisfies EntitlementReader ? my prediction: ___   # TODO: predict BEFORE running

print("chain satisfies NetworkProvisioner?", isinstance(chain, NetworkProvisioner))
print("net   satisfies EntitlementReader ?", isinstance(net, EntitlementReader))
# TODO: un-comment the self-check once your predictions are written down
# assert not isinstance(chain, NetworkProvisioner) and not isinstance(net, EntitlementReader)

<details><summary>✅ Solution 3.1 — peek only after trying</summary>

```python
assert not isinstance(chain, NetworkProvisioner)   # no apply_bandwidth / teardown / health
assert not isinstance(net, EntitlementReader)      # no owner_of / get / chain_time / watch_revoked
```

Structural typing checks method names, and each prop carries exactly its own port's methods — the fit is precise, not accidental.

</details>

## 4. `fulfill()` — every check, then every write

`fulfill` is the single on-chain moment of the whole story: salt punched, 10 TOK moved,
ticket #7 minted — *or nothing at all*. That "or nothing" is called **atomicity**
(*atomic / all-or-nothing* — either every effect happens or none does). Read the source
top to bottom and notice two things:

1. **The check order**: expired → consumer binding → salt → funds. That order is not
   style — it is the *spec* the real Solidity contract honors, revert for revert
   (chapters 06 and 07 hold this fake against the real thing).
2. **Every `raise` comes before the first write.** The real chain gets all-or-nothing
   for free: a revert rolls back every storage write in the transaction. Python has no
   rollback, so this fake *earns* atomicity by ordering alone.

In [ ]:
print(inspect.getsource(FakeChain.fulfill))

Run the happy path. We hand it the canonical signed offer directly (the agents who
*produce* it are §6's topic) and watch all four effects land at once: balances move,
salt punched, ticket minted, owner recorded.

In [ ]:
signed = fx.CANONICAL_SIGNED_OFFER
ada_before, bell_before = chain.balances[fx.ADA], chain.balances[fx.BELL]

eid = chain.fulfill(signed, buyer=fx.ADA)

print("fulfill() ->", eid, " ← the new ticket id")
print("Ada :", ada_before // 10**18, "->", chain.balances[fx.ADA] // 10**18, "TOK")
print("Bell:", bell_before // 10**18, "->", chain.balances[fx.BELL] // 10**18, "TOK")
print("salt punched?", signed.offer.salt in chain.consumed)
print("owner of #7 :", chain.owner_of(eid)[:8] + "…  == Ada?", chain.owner_of(eid) == fx.ADA)
assert eid == fx.TICKET_ID
assert chain.balances[fx.ADA] == ada_before - int(fx.PRICE_10_TOK)
assert chain.balances[fx.BELL] == bell_before + int(fx.PRICE_10_TOK)
assert signed.offer.salt in chain.consumed and chain.owner_of(eid) == fx.ADA

The mint stored an `EntitlementView` — ticket #7 as the controller will later read it.
Its `params` field is no longer the opaque ABI blob you dissected by hand in 04: the
fake decoded it with `_decode_bandwidth_params`, two `int(x, 16)` slices standing in
for chainmcp's real decoder (rule 2 keeps real codecs — and keys — out of here).

In [ ]:
from e2e.skeleton.fakes import _decode_bandwidth_params  # _private: peeking to learn

view = chain.get(eid)
print("id      :", view.id, "  issuer:", view.issuer[:8] + "… (Bell)")
print("window  :", hhmm(view.start_time), "->", hhmm(view.end_time))
print("params  :", view.params)
print("revoked :", view.revoked)

decoded = _decode_bandwidth_params(fx.BANDWIDTH_PARAMS_ABI)
assert decoded == view.params and decoded.capacity_bps == fx.CAPACITY_50_MBPS
print("blob …" + fx.BANDWIDTH_PARAMS_ABI[-20:], "->", decoded, " ← 04's blob, decoded")

**✏️ Your turn 4.1 — Ask for the ticket that was never minted**

`fulfill()` wrote exactly four effects — so ticket #8 must not exist. Ask the chain who owns `fx.TICKET_ID + 1` and read the failure. Success: you catch a `KeyError` carrying the number 8 — a raw dict miss, because no `ErrorCode` names "no such ticket" on this fake.

In [ ]:
caught = None
try:
    pass  # TODO: replace `pass` with: chain.owner_of(fx.TICKET_ID + 1)
except KeyError as e:
    caught = e
print("caught:", type(caught).__name__ if caught else "nothing yet — edit the TODO")
# TODO: un-comment once the error is caught
# assert isinstance(caught, KeyError) and caught.args[0] == fx.TICKET_ID + 1

<details><summary>✅ Solution 4.1 — peek only after trying</summary>

```python
caught = None
try:
    chain.owner_of(fx.TICKET_ID + 1)
except KeyError as e:
    caught = e
assert isinstance(caught, KeyError) and caught.args[0] == fx.TICKET_ID + 1
```

`_owners` is a plain dict, so "never minted" surfaces as the container's own `KeyError` — the real chain reverts instead, as chapter 06 shows.

</details>

## 5. The four deny paths — and the no-trace proof

Now break `fulfill` on purpose, once per check, in the contract's order. The claim
worth proving each time is not just "it raised the right error" — it is that the
rejected fulfill **left no trace**: salt unpunched, no money moved, nothing minted.
That is §4's hand-made atomicity doing its job.

Two helpers keep the cells honest: a fresh stage per attempt, and one function that
asserts the stage is *exactly* as seeded.

In [ ]:
def new_chain(now: int = STORY_TIME) -> tuple[FakeClock, FakeChain]:
    """A fresh stage: Ada 50 TOK, Bell 0, next ticket #7 — §3's seeding, repeatable."""
    c = FakeClock(now)
    return c, FakeChain(c, balances={fx.ADA: SEED, fx.BELL: 0}, next_id=fx.TICKET_ID)


def assert_no_trace(ch: FakeChain, offer_salt: str) -> None:
    """The atomicity claim: a rejected fulfill changed NOTHING."""
    assert offer_salt not in ch.consumed, "salt was punched!"
    assert ch.balances[fx.ADA] == SEED and ch.balances[fx.BELL] == 0, "money moved!"
    try:
        ch.owner_of(fx.TICKET_ID)
        raise AssertionError("a ticket was minted!")
    except KeyError:
        pass
    print("   no trace ✓  (salt unpunched, balances intact, nothing minted)")


print("helpers ready")

**Deny 1 — `OfferExpired`.** The offer's `valid_until` is 14:20: quotes go stale fast
(the *service window* is the different, longer thing). Fast-forward chain time one
second past it and try to pay.

In [ ]:
clk1, ch1 = new_chain()
clk1.advance(signed.offer.valid_until - clk1.now() + 1)      # one second too late
print(f"chain time {clk1.now()} = valid_until {signed.offer.valid_until} ({hhmm(signed.offer.valid_until)}) + 1 s")

try:
    ch1.fulfill(signed, buyer=fx.ADA)
    raise AssertionError("should have raised")
except OfferExpired as e:
    print("✗ OfferExpired:", e, " ← the valid_until it died on")
assert_no_trace(ch1, signed.offer.salt)

**Deny 2 — `WrongConsumer`.** The canonical offer is *open* (`consumer` = the zero
address: anyone may redeem it). Bell can instead write a buyer's name on the ticket.
The shapes are frozen (02), so "editing" an offer means `model_copy` — a brand-new
object. And note what the fake *cannot* check: on the real chain, retargeting an
already-signed offer breaks the signature (`BadSignature`); this fake never verifies
signatures at all. That honesty gap closes in 07.

In [ ]:
targeted = signed.model_copy(
    update={"offer": signed.offer.model_copy(update={"consumer": fx.ADA})}
)
print("offer now bound to:", targeted.offer.consumer[:8] + "… (Ada)")

clk2, ch2 = new_chain()
try:
    ch2.fulfill(targeted, buyer=fx.BELL)                     # Bell grabs Ada's ticket
    raise AssertionError("should have raised")
except WrongConsumer as e:
    print("✗ WrongConsumer:", str(e)[:8] + "…", " ← the impostor's address")
assert_no_trace(ch2, targeted.offer.salt)

assert ch2.owner_of(ch2.fulfill(targeted, buyer=fx.ADA)) == fx.ADA
print("   …and the rightful Ada still can ✓")

**Deny 3 — `OfferAlreadyUsed`.** Pay once (fine), then replay the *same* signed offer.
The salt — the offer's one-of-a-kind serial number, from 04 — is already in the
`consumed` set, so the second attempt dies. The trace check here is relative: Bell was
paid exactly once, and no ticket #8 exists.

In [ ]:
clk3, ch3 = new_chain()
first = ch3.fulfill(signed, buyer=fx.ADA)
bell_after_one = ch3.balances[fx.BELL]

try:
    ch3.fulfill(signed, buyer=fx.ADA)                        # the exact same offer again
    raise AssertionError("should have raised")
except OfferAlreadyUsed:
    print("✗ OfferAlreadyUsed: salt …" + signed.offer.salt[-4:], "is already punched")

assert ch3.balances[fx.BELL] == bell_after_one               # paid once, not twice
try:
    ch3.owner_of(first + 1)
    raise AssertionError("a second ticket was minted!")
except KeyError:
    print("   Bell paid once ✓  no ticket #8 ✓")

**Deny 4 — `InsufficientFunds`.** Meet Mallory — the story's freeloader, an account
with gas money but zero TOK. Her address is a constant in `worlds.py` (chapter 11's
file; we only borrow the name). She grabs the open offer without the 10 TOK to pay it.

In [ ]:
from e2e.skeleton.worlds import MALLORY

clk4, ch4 = new_chain()
print("Mallory's TOK:", ch4.balances.get(MALLORY, 0), " ← .get(addr, 0): absent means broke")

try:
    ch4.fulfill(signed, buyer=MALLORY)
    raise AssertionError("should have raised")
except InsufficientFunds as e:
    print("✗ InsufficientFunds:", str(e)[:8] + "…")
assert_no_trace(ch4, signed.offer.salt)

**✏️ Your turn 5.1 — Give Ada exactly the price**

Deny 4 fired because Mallory's balance was *less than* the price — the check reads `balance < price`, strictly. Re-seed the stage so Ada holds exactly 10 TOK (`int(fx.PRICE_10_TOK)`) and observe: refusal or ticket? Success: the fulfill succeeds and Ada walks away with exactly 0 TOK.

In [ ]:
clk6 = FakeClock(STORY_TIME)
exact = FakeChain(clk6, balances={fx.ADA: SEED, fx.BELL: 0}, next_id=fx.TICKET_ID)
# TODO: re-seed `exact` above so Ada holds EXACTLY the price: {fx.ADA: int(fx.PRICE_10_TOK), fx.BELL: 0}

outcome = exact.fulfill(signed, buyer=fx.ADA)
print("fulfill → ticket", outcome, "| Ada left with", exact.balances[fx.ADA] // 10**18, "TOK")
# TODO: un-comment once Ada is seeded with exactly 10 TOK
# assert outcome == fx.TICKET_ID and exact.balances[fx.ADA] == 0

<details><summary>✅ Solution 5.1 — peek only after trying</summary>

```python
clk6 = FakeClock(STORY_TIME)
exact = FakeChain(clk6, balances={fx.ADA: int(fx.PRICE_10_TOK), fx.BELL: 0}, next_id=fx.TICKET_ID)
outcome = exact.fulfill(signed, buyer=fx.ADA)
assert outcome == fx.TICKET_ID and exact.balances[fx.ADA] == 0
```

`<` (not `<=`) means "exactly enough" pays — the same boundary the real ERC-20 transfer honors in 06.

</details>

**One more thing the order buys.** Stage an attempt that fails *all four* checks at
once: expired clock, offer bound to Ada, salt pre-punched by hand, and broke Mallory as
the buyer. Four true reasons to refuse — the answer is the **first** one in the order.
When you read the real contract's revert order in 06, this is the exact behavior it
must reproduce.

In [ ]:
clk5, ch5 = new_chain(now=signed.offer.valid_until + 1)      # expired…
ch5.consumed.add(targeted.offer.salt)                        # …salt pre-punched by hand…

try:
    ch5.fulfill(targeted, buyer=MALLORY)                     # …wrong consumer AND broke
    raise AssertionError("should have raised")
except OfferExpired:
    print("four reasons to say no -> OfferExpired  ← the FIRST check answers")
except (WrongConsumer, OfferAlreadyUsed, InsufficientFunds) as e:
    raise AssertionError(f"wrong check answered first: {type(e).__name__}")

## 6. Scripted agents — the judgment slots, hardcoded

Where did the `SignedOffer` we have been spending come from? In the finished system,
from **judgment**: rule 1 says LLM judgment exists in exactly two places — the
provider's quote/decline and the consumer's accept/reject. Everything else is
deterministic code. The skeleton keeps both *slots* but cans the answers, so the play
is reproducible forever (an LLM is slow, costs money, and answers differently each run
— useless for a test that must pass identically in CI). The whole file fits on one
screen:

In [ ]:
import e2e.skeleton.scripted_agents as scripted

print(inspect.getsource(scripted))

Poke both slots. Two things to notice: the provider returns *the* shared fixture object
(`is`, not merely `==` — no copy is made), and the signature is a 65-byte `0xabab…`
placeholder — real signing lives only in chainmcp, the sole key holder (rule 2; chapter
07). The `need` argument is ignored entirely: even `quote(None)` works.

In [ ]:
off = ScriptedProvider().quote(None)                 # need ignored — it's a script
assert off is fx.CANONICAL_SIGNED_OFFER              # THE fixture object, not a copy
print("quote(None) -> the canonical offer:", int(off.offer.price) // 10**18, "TOK for",
      off.offer.end_time - off.offer.start_time, "s of bandwidth")
print("signature   :", off.signature[:12] + "…", " ← placeholder; nothing here can sign (rule 2)")

d = ScriptedConsumer().decide(None, off)
assert d.accept
print("decide()    ->", d.model_dump(), " ← always yes; judgment arrives in chapter 10")

Handing every caller one shared singleton would be reckless — except the shapes are
**frozen** pydantic models (02). Break it on purpose: try to lower the price on the
shared offer.

In [ ]:
try:
    fx.CANONICAL_SIGNED_OFFER.offer.price = "0"
    raise AssertionError("mutation should be impossible")
except pydantic.ValidationError as e:
    print("✗", e.errors()[0]["type"], "->", e.errors()[0]["msg"], " ← frozen model (02)")
print("price still:", int(fx.CANONICAL_SIGNED_OFFER.offer.price) // 10**18, "TOK ✓")

**✏️ Your turn 6.1 — Discount the offer the legal way**

You just saw mutation refused. `model_copy(update=...)` is the legal edit: it builds a brand-new object and leaves the shared original alone. Make a 5 TOK twin of the canonical offer. Success: the two prices print 5 and 10, and the self-check passes.

In [ ]:
cheaper = off.offer    # TODO: replace with off.offer.model_copy(update={"price": str(5 * 10**18)})

print("copy's price   :", int(cheaper.price) // 10**18, "TOK")
print("original price :", int(fx.CANONICAL_SIGNED_OFFER.offer.price) // 10**18, "TOK  ← must stay 10")
# TODO: un-comment once the copy is discounted
# assert int(cheaper.price) == 5 * 10**18 and int(fx.CANONICAL_SIGNED_OFFER.offer.price) == 10 * 10**18

<details><summary>✅ Solution 6.1 — peek only after trying</summary>

```python
cheaper = off.offer.model_copy(update={"price": str(5 * 10**18)})
assert int(cheaper.price) == 5 * 10**18
assert int(fx.CANONICAL_SIGNED_OFFER.offer.price) == 10 * 10**18
```

Frozen models make sharing safe and edits explicit — and on the real chain this cheaper twin would still die at 07's signature check, exactly like §5's retargeted offer.

</details>

## 7. The predicate — the bouncer's checklist

Ada now owns ticket #7. Owning it is not the same as *getting the service*: at the
door, someone must decide "does THIS requester get THIS service NOW?" That decision is
the **predicate**, and it is rule 1's other half: authorization is deterministic code,
never an LLM. A creative bouncer can be sweet-talked; this one is boring arithmetic.

The skeleton does not own a private copy. It imports the REAL predicate from
`controller.domain` — exactly one bouncer exists in the whole codebase — and merely
narrows its scope. First, the skeleton's thin wrapper:

In [ ]:
print(inspect.getsource(predicate))

`known_service_types=(0,)` is honesty: this stub can only translate *bandwidth* into
router calls. Admitting a telemetry ticket would pass the checklist and then crash
mid-provision — better to refuse at the door (`E_SCOPE`). Now the real checklist it
delegates to, from `controller/domain.py`. (Chapter 08 dissects its home — and proves
the module imports no I/O, rule 4. Here we read the function itself.)

In [ ]:
from controller import domain

print(inspect.getsource(domain.predicate))

Walk the checks in order — the order is contract, "who" before state:

1. `requester != owner` → `E_NOT_OWNER` — only the ticket's *current owner* may activate
2. `now < start_time` → `E_NOT_STARTED` — the window has not opened
3. `now >= end_time` → `E_EXPIRED` — the window is over (end is exclusive)
4. `view.revoked` → `E_REVOKED` — the issuer pulled the ticket
5. unknown `service_type` → `E_SCOPE` — no translator, refuse at the door
6. `view.id in active_ids` → `E_CONFLICT` — one ticket, one live session at a time

Beginner trap: **`None` means "allowed."** The function returns the first problem
found, or `None` when there is nothing to complain about. It is also *pure*: it never
fetches anything — `owner`, `now`, `active_ids` are handed in — which is why we can
call it right here with no chain in sight. Accept case first, at 14:02:

In [ ]:
V = fx.CANONICAL_ENTITLEMENT_VIEW           # ticket #7 exactly as minted (04)
t_1402 = fx.WINDOW.start + 120              # 14:02 — two minutes into the window

verdict = predicate(V, owner=fx.ADA, requester=fx.ADA, now=t_1402, active_ids=set())
print("Ada, 14:02, no live sessions ->", verdict, " ← None means ALLOWED")
assert verdict is None

Now flip **one fact per run** and collect every denial with its `ErrorCode`. The
revoked and telemetry variants need `model_copy` (frozen models, 02). We call the
stub's wrapper, so `service_type=1` lands in `E_SCOPE` — the wrapper's scope is
bandwidth-only.

In [ ]:
flips = [
    ("wrong requester", predicate(V, fx.ADA, fx.BELL, t_1402, set()), ErrorCode.E_NOT_OWNER),
    ("13:59, too early", predicate(V, fx.ADA, fx.ADA, fx.WINDOW.start - 1, set()), ErrorCode.E_NOT_STARTED),
    ("16:00, too late ", predicate(V, fx.ADA, fx.ADA, fx.WINDOW.end, set()), ErrorCode.E_EXPIRED),
    ("ticket revoked  ", predicate(V.model_copy(update={"revoked": True}), fx.ADA, fx.ADA, t_1402, set()), ErrorCode.E_REVOKED),
    ("can't translate ", predicate(V.model_copy(update={"service_type": 1}), fx.ADA, fx.ADA, t_1402, set()), ErrorCode.E_SCOPE),
    ("already active  ", predicate(V, fx.ADA, fx.ADA, t_1402, {V.id}), ErrorCode.E_CONFLICT),
]
for label, got, want in flips:
    assert got == want, (label, got)
    print(f"✗ {label} -> {got.value}")

**✏️ Your turn 7.1 — Call the boundary second**

The flips table refused 16:00:00 with `E_EXPIRED` — check 3 reads `now >= end_time`, an *exclusive* end. Predict the verdict at 15:59:59 (`fx.WINDOW.end - 1`) before running. Success: one `None` and one `E_EXPIRED`, exactly one second apart.

In [ ]:
# my prediction at 15:59:59: ___     # TODO: predict BEFORE running
# my prediction at 16:00:00: ___

last_second = predicate(V, owner=fx.ADA, requester=fx.ADA, now=fx.WINDOW.end - 1, active_ids=set())
stroke_of_16 = predicate(V, owner=fx.ADA, requester=fx.ADA, now=fx.WINDOW.end, active_ids=set())
print("15:59:59 →", last_second, "   16:00:00 →", stroke_of_16)
# TODO: un-comment after writing your predictions
# assert last_second is None and stroke_of_16 == ErrorCode.E_EXPIRED

<details><summary>✅ Solution 7.1 — peek only after trying</summary>

```python
assert predicate(V, fx.ADA, fx.ADA, fx.WINDOW.end - 1, set()) is None
assert predicate(V, fx.ADA, fx.ADA, fx.WINDOW.end, set()) == ErrorCode.E_EXPIRED
```

`>=` makes 15:59:59 the last paid second — the same exclusivity §9's `tick()` enforces when it tears the session down at exactly 16:00.

</details>

Order matters here too. Hand the bouncer a request that fails *everything* — wrong
requester, before the window, revoked ticket, conflicting session — and it reports the
**first** failing check, nothing else. Callers (and tests, and chapter 08's golden
files) rely on that determinism.

In [ ]:
mess = V.model_copy(update={"revoked": True})
verdict = predicate(mess, owner=fx.ADA, requester=fx.BELL,
                    now=fx.WINDOW.start - 999, active_ids={mess.id})
assert verdict == ErrorCode.E_NOT_OWNER
print("four facts wrong ->", verdict.value, " ← only the FIRST check answers")

## 8. `StubController` — the bouncer, employed

The predicate answers questions; something must gather the facts, ask it, and act on
the answer. That is `StubController`. Its constructor takes the two **ports** — not a
`FakeChain` but an `EntitlementReader`, not a `FakeNet` but a `NetworkProvisioner`.
Handing dependencies in instead of building them inside is called **dependency
injection**, and it is the entire reason cardboard works today and real adapters drop
in later, unchanged.

One subtle line in `__init__`: it hands the chain a **callback** — a function the chain
will call *back* when a ticket is revoked (the usual direction inverts: the chain calls
the controller). It sits silent until §10. Build a fresh stage and check the callback
got registered:

In [ ]:
clk_s, chain_s = new_chain()                 # 13:32, Ada funded, next ticket #7
net_s = FakeNet()
ctrl = StubController(chain_s, net_s)

eid = chain_s.fulfill(fx.CANONICAL_SIGNED_OFFER, buyer=fx.ADA)
assert eid == fx.TICKET_ID
print("ticket #7 minted; controller built")
print("chain watchers:", len(chain_s._watchers), " ← __init__ registered _on_revoked")
assert len(chain_s._watchers) == 1

Activation is a two-step dance. Step one: `challenge()` — the controller mints a
**nonce** (*number used once*): a fresh one-time pass it will demand back and then
burn. This is replay protection in embryo; chapter 08 builds the full
challenge–response with signed proofs. Here, the nonce *is* the proof.

In [ ]:
nonce = ctrl.challenge(eid)
print("challenge() ->", repr(nonce))
print("open nonces :", ctrl._open_nonces, " ← peeking at internals to learn")
assert nonce in ctrl._open_nonces

Step two: `activate()`. Read it as three beats — **fetch** (ask the chain for the
ticket, the owner, the time), **decide** (the pure predicate), **act** (resolve the
resource, call the router, record the session). All I/O at the edges, one pure decision
in the middle: that shape *is* the architecture.

In [ ]:
print(inspect.getsource(StubController.activate))

The `_resolve` call is the paper-to-physics moment: the ticket's opaque `resource_id`
becomes a concrete device and interfaces. In v0 the map is a one-entry dict — the
stand-in for `resource_map.yaml`, the ONLY place in the system allowed to know topology
(rule 6 / ADR-005; the real YAML file arrives in chapter 08).

In [ ]:
from e2e.skeleton.stub_controller import _RESOURCE_MAP  # _private: peeking to learn

for rid, path in _RESOURCE_MAP.items():
    print(f"resource_id …{rid.hex()[-4:]} -> {path.device}: {path.ingress_if} -> {path.egress_if}")
assert _RESOURCE_MAP[bytes.fromhex(fx.RESOURCE_ID[2:])] == fx.RESOLVED_PATH

Now the trap every demo of this repo hits once: our stage opened at **13:32**, and
Ada's window starts at **14:00**. Activate right now and the bouncer says *not yet* —
and notice the nonce is burned anyway. The burn happens *before* the checklist runs, so
even a denied attempt consumes the pass.

In [ ]:
try:
    ctrl.activate(eid, requester=fx.ADA, nonce=nonce)
    raise AssertionError("should have been denied")
except Denied as e:
    print(f"✗ activate at {hhmm(clk_s.now())} ->", e.code.value, " ← window opens at 14:00")
    assert e.code == ErrorCode.E_NOT_STARTED

assert nonce not in ctrl._open_nonces
print("the nonce is gone:", ctrl._open_nonces, " ← burned even on a denial")

**✏️ Your turn 8.1 — Forge a pass**

Run the cell once as-is: a *legitimate* nonce travels all the way to the checklist and dies on the window (`E_NOT_STARTED`). Now forge: replace it with an invented string like `"nonce-999"` and rerun. Success: the refusal changes to `E_NONCE_REUSED` — the forged pass never even reaches the checklist.

In [ ]:
sneaky_nonce = ctrl.challenge(eid)    # TODO: replace this legit pass with a forged one, e.g. "nonce-999"

refusal = None
try:
    ctrl.activate(eid, requester=fx.ADA, nonce=sneaky_nonce)
    print("activate let it through?!")
except Denied as e:
    refusal = e.code
    print("refused →", refusal.value)
# TODO: un-comment once the nonce is forged
# assert refusal == ErrorCode.E_NONCE_REUSED  # the set can't tell "never issued" from "used up"

<details><summary>✅ Solution 8.1 — peek only after trying</summary>

```python
refusal = None
try:
    ctrl.activate(eid, requester=fx.ADA, nonce="nonce-999")
except Denied as e:
    refusal = e.code
assert refusal == ErrorCode.E_NONCE_REUSED
```

A set can't distinguish "never issued" from "already burned", so both refusals share one code — and proof-before-predicate is the order that keeps forged passes away from the checklist entirely.

</details>

Advance chain time to 14:02, take a *fresh* nonce, and knock again. This time every
fact checks out, so the controller resolves the path, calls the provisioner, and
records an ACTIVE session.

In [ ]:
clk_s.advance((fx.WINDOW.start + 120) - clk_s.now())        # -> 14:02
sid = ctrl.activate(eid, requester=fx.ADA, nonce=ctrl.challenge(eid))
print(f"{hhmm(clk_s.now())}  activate() -> {sid!r}, state = {ctrl.state(sid).value}")
assert ctrl.state(sid) == SessionState.ACTIVE

What did "provisioning" actually do? Nothing touched a router — `FakeNet` just wrote
down the request. Inspect exactly what it recorded: the device, the path, and the rate.
These very fields become a gNMI `Set` on a real SR Linux box in chapter 09.

In [ ]:
cfg = net_s.applied[sid]
print("device :", cfg["path"].device)
print("path   :", cfg["path"].ingress_if, "->", cfg["path"].egress_if)
print("rate   :", f"{cfg['capacity_bps'] // 1000:,} kbps", f"(= {cfg['capacity_bps'] // 1_000_000} Mbps)")
print("qos    :", cfg["qos_class"])
assert cfg["path"] == fx.RESOLVED_PATH
assert cfg["capacity_bps"] == fx.CAPACITY_50_MBPS and cfg["qos_class"] == fx.QOS_CLASS

Three more knocks on the door, each refused for its own reason: the burnt nonce again
(replay), Bell with a fresh nonce (signing the offer ≠ owning the ticket), and Ada
herself trying to double-book the ticket she is already using.

In [ ]:
try:
    ctrl.activate(eid, requester=fx.ADA, nonce=nonce)       # the OLD burnt nonce
    raise AssertionError("should have been denied")
except Denied as e:
    assert e.code == ErrorCode.E_NONCE_REUSED
    print("✗ replayed nonce   ->", e.code.value)

try:
    ctrl.activate(eid, requester=fx.BELL, nonce=ctrl.challenge(eid))
    raise AssertionError("should have been denied")
except Denied as e:
    assert e.code == ErrorCode.E_NOT_OWNER
    print("✗ Bell knocks      ->", e.code.value, " ← issuer ≠ owner")

try:
    ctrl.activate(eid, requester=fx.ADA, nonce=ctrl.challenge(eid))
    raise AssertionError("should have been denied")
except Denied as e:
    assert e.code == ErrorCode.E_CONFLICT
    print("✗ Ada double-books ->", e.code.value, " ← one ticket, one live session")

## 9. Teardown path A — the clock runs out

Ada bought 14:00 → 16:00, not forever. `tick()` is the time-driven ending: something
wakes the controller periodically (an OS timer may do the *waking* — ADR-004 allows
that), but the *decision* re-reads `chain_time()` and compares it to each ticket's
`end_time`. The wake-up schedules; the chain decides. Read both `tick` and the
`teardown` it funnels into:

In [ ]:
print(inspect.getsource(StubController.tick))
print(inspect.getsource(StubController.teardown))

Jump chain time to exactly 16:00 and tick. Three things to watch: the session state
flips, the config disappears from `net.applied`, and the teardown call is recorded in
`net.torn_down`.

In [ ]:
clk_s.advance(fx.WINDOW.end - clk_s.now())   # jump to exactly 16:00
ctrl.tick()
print(f"{hhmm(clk_s.now())}  tick() -> session {ctrl.state(sid).value}")
print("net.applied  :", net_s.applied, " ← config removed")
print("net.torn_down:", net_s.torn_down)
assert ctrl.state(sid) == SessionState.TORN_DOWN
assert sid in net_s.torn_down and sid not in net_s.applied

**Rule 8: teardown is idempotent** — *idempotent* meaning doing it again changes
nothing and still succeeds. Why insist? Because this section and the next are two
*independent* paths to the same teardown, and both can fire on the same session; if the
second call errored, correct behavior would look like a crash. Call everything again —
twice — and watch nothing break. `FakeNet` records every call (`pop` with a default
never raises), so the idempotency is *observable*, not just claimed.

In [ ]:
ctrl.tick()                                  # nothing left to expire — fine
ctrl.teardown(sid)                           # tearing down what's already down — fine
ctrl.teardown(sid)                           # ...and again — still fine
print("state         :", ctrl.state(sid).value)
print("torn_down list:", net_s.torn_down, " ← every call recorded, none failed")
assert ctrl.state(sid) == SessionState.TORN_DOWN
assert net_s.torn_down.count(sid) == 3       # tick's teardown + our two extras
print("✓ rule 8: tearing down twice is a success, not an error")

**✏️ Your turn 9.1 — Tear down a ghost**

Rule 8's sharpest edge: point `teardown` at a session id that never existed. Add the call, rerun, and watch what `net.torn_down` records. Success: no exception, and the ghost id appears in the list — refusing would be the bug.

In [ ]:
before = list(net_s.torn_down)
# TODO: call ctrl.teardown("sess-ghost") — a session id that never existed

print("torn_down before:", before)
print("torn_down after :", net_s.torn_down)
# TODO: un-comment once you've made the ghost call
# assert net_s.torn_down == before + ["sess-ghost"]  # recorded, and nothing raised

<details><summary>✅ Solution 9.1 — peek only after trying</summary>

```python
before = list(net_s.torn_down)
ctrl.teardown("sess-ghost")
assert net_s.torn_down == before + ["sess-ghost"]
```

`teardown` pops with a default and only flips state if the session exists — "already gone" and "never existed" are both success (rule 8), because a crash here would look like a stuck session.

</details>

## 10. Teardown path B — Bell pulls the ticket (the showpiece)

The scarier ending: the session is live, mid-window, and the *issuer revokes the
entitlement*. A polling loop would not notice until its next wake-up; instead the chain
**pushes** — `revoke()` rings every callback registered via `watch_revoked`, and our
controller registered one at birth (§8). Read the two sides of the phone call, and
notice two disciplines: the controller *re-reads the ticket* before acting (the event
is a nudge; the chain is the truth — ADR-004), and revocation is a **flag flip, never a
delete** (invariant I5: a revoked ticket must stay readable as evidence).

In [ ]:
print(inspect.getsource(FakeChain.revoke))
print(inspect.getsource(StubController._on_revoked))

Fresh stage, curtain up at 14:02 so we can activate immediately — a brand-new clock,
chain, net, and controller, so nothing from §9 leaks in.

In [ ]:
clk_r, ch_r = new_chain(now=fx.WINDOW.start + 120)          # curtain up at 14:02
net_r = FakeNet()
ctrl_r = StubController(ch_r, net_r)

eid_r = ch_r.fulfill(fx.CANONICAL_SIGNED_OFFER, buyer=fx.ADA)
sid_r = ctrl_r.activate(eid_r, requester=fx.ADA, nonce=ctrl_r.challenge(eid_r))
assert ctrl_r.state(sid_r) == SessionState.ACTIVE
print(f"{hhmm(clk_r.now())}  session {sid_r!r} ACTIVE — window runs to {hhmm(fx.WINDOW.end)}")

At 15:10 — fifty minutes of paid window still left — Bell pulls ticket #7. One call.
No `tick()`, no waiting: the callback chain fires *inside* `revoke()` and the session
is down before the next line of this cell runs.

In [ ]:
clk_r.advance((fx.WINDOW.start + 4200) - clk_r.now())       # -> 15:10, mid-window
ch_r.revoke(eid_r)                                          # Bell pulls the ticket

print(f"{hhmm(clk_r.now())}  revoke(#{eid_r}) -> callback -> session {ctrl_r.state(sid_r).value}  ← no tick()")
assert ctrl_r.state(sid_r) == SessionState.TORN_DOWN
assert sid_r in net_r.torn_down and sid_r not in net_r.applied

view_r = ch_r.get(eid_r)
assert view_r.revoked and view_r.id == fx.TICKET_ID
print("ticket #7 still readable, revoked =", view_r.revoked, " ← flag flip, never a burn (I5)")

**✏️ Your turn 10.1 — Revoke the already-revoked ticket**

Bell panics and pulls ticket #7 a *second* time. Predict before running: crash, a second teardown, or nothing at all? Success: your prediction is written down and the counter tells the truth.

In [ ]:
# second revoke — crash / second teardown / nothing? my prediction: ___   # TODO: predict first

count_before = net_r.torn_down.count(sid_r)
ch_r.revoke(eid_r)                    # Bell pulls the already-pulled ticket
count_after = net_r.torn_down.count(sid_r)
print("teardowns for the session:", count_before, "→", count_after)
print("still revoked:", ch_r.get(eid_r).revoked, "| session:", ctrl_r.state(sid_r).value)
# TODO: un-comment after writing your prediction
# assert count_after == count_before  # the callback re-checked: no ACTIVE session, nothing to do

<details><summary>✅ Solution 10.1 — peek only after trying</summary>

```python
ch_r.revoke(eid_r)                    # fires the callbacks again
assert net_r.torn_down.count(sid_r) == 1
```

The callback re-reads the truth and finds no ACTIVE session left, so it does nothing — contrast with 9.1, where calling `teardown` directly always records a call.

</details>

You will see this exact scene again in `12_grand_finale.ipynb` — with a real on-chain
event picked up by a real watcher instead of a Python list of callbacks. Cardboard
first, so you recognize the shape when it is real.

## 11. The whole play, one cell

Everything above, end to end, narrated like the story's epilogue. This is the same
script `e2e/tests/test_lifecycle.py` runs in CI on every commit (run it with
`pytest -s` and you'll see these same lines) — which is the walking skeleton's real
job: if any later milestone miswires a joint, this play stops mid-scene.

In [ ]:
play_clk, play_chain = new_chain()                          # 13:32
play_net = FakeNet()
play_ctrl = StubController(play_chain, play_net)
provider, consumer = ScriptedProvider(), ScriptedConsumer()

need = fx.BANDWIDTH_NEED
print(f"{hhmm(play_clk.now())}  Ada needs {need.capacity_bps // 1_000_000} Mbps {need.src}->{need.dst}")

offer = provider.quote(need)                                # Bell's judgment slot (canned)
decision = consumer.decide(need, offer)                     # Ada's judgment slot (canned)
assert decision.accept
print(f'{hhmm(play_clk.now())}  Bell quotes 50 Mbps / 10 TOK; Ada: {{"accept": {decision.accept}}}')

eid = play_chain.fulfill(offer, buyer=fx.ADA)               # the one on-chain write
assert eid == fx.TICKET_ID and play_chain.owner_of(eid) == fx.ADA
print(f"{hhmm(play_clk.now())}  fulfill(): ticket #{eid} -> Ada, 10 TOK -> Bell, salt punched")

play_clk.advance(1800)                                      # -> 14:02, inside the window
sid = play_ctrl.activate(eid, requester=fx.ADA, nonce=play_ctrl.challenge(eid))
assert play_ctrl.state(sid) == SessionState.ACTIVE
kbps = play_net.applied[sid]["capacity_bps"] // 1000
print(f"{hhmm(play_clk.now())}  checklist passed; provision: police {kbps:,} kbps")

play_clk.advance(play_chain.get(eid).end_time - play_clk.now())  # jump to exactly 16:00
play_ctrl.tick()
assert play_ctrl.state(sid) == SessionState.TORN_DOWN and sid in play_net.torn_down
print(f"{hhmm(play_clk.now())}  chain time >= end_time -> torn down")
print()
print("the whole play, cardboard props, one cell ✓")

## 12. Break it on purpose — what makes single-use *real*?

§5's replay was refused because of one line in `fulfill`:
`if offer.salt in self.consumed`. Is that line load-bearing, or would honest buyers
behave anyway? Sabotage the ledger and find out: a subclass whose `fulfill` *forgets*
each salt just before checking — exactly what deleting the check would do.

In [ ]:
class AmnesiacChain(FakeChain):
    """FakeChain with the single-use ledger sabotaged: every salt is forgotten
    the moment it matters — as if the `salt in consumed` check were deleted."""

    def fulfill(self, signed: object, buyer: str) -> int:
        self.consumed.discard(signed.offer.salt)    # wipe the memory, then check
        return super().fulfill(signed, buyer)


amn_clk = FakeClock(STORY_TIME)
evil = AmnesiacChain(amn_clk, balances={fx.ADA: SEED, fx.BELL: 0}, next_id=fx.TICKET_ID)

first = evil.fulfill(fx.CANONICAL_SIGNED_OFFER, buyer=fx.ADA)
second = evil.fulfill(fx.CANONICAL_SIGNED_OFFER, buyer=fx.ADA)  # the SAME signed offer
print("minted:", first, "and", second, " ← one signature, two tickets")
print("Ada :", evil.balances[fx.ADA] // 10**18, "TOK  (charged twice)")
print("Bell:", evil.balances[fx.BELL] // 10**18, "TOK  (collected twice)")
assert (first, second) == (7, 8)
assert evil.balances[fx.ADA] == SEED - 2 * int(fx.PRICE_10_TOK)

One signature, two tickets, Ada charged twice — and nothing else objected. Not the
shapes, not the agents, not the signature: a signature can be *copied* perfectly, which
is precisely what a replay is. **Single-use is a ledger, not politeness.** It exists
only because the chain remembers every spent salt and refuses repeats. On the real
contract that memory is a storage mapping and the refusal is a revert — invariant I2,
pinned by a Foundry test literally named `test_I2_offerSingleUse`. Chapter 06 shows you
the very line.

**✏️ Your turn 12.1 — Sabotage a different check**

You just proved single-use is a ledger. Now prove the funds check is load-bearing the same way: finish `CharityChain`, whose `fulfill` tops the buyer up to the price first — exactly what deleting `balance < price` would buy. Success: broke Mallory walks out owning ticket #7 and Bell is "paid" with money that appeared from nowhere.

In [ ]:
class CharityChain(FakeChain):
    """FakeChain with the funds check neutered: every buyer is topped up just
    before the check — as if `balance < price` were deleted."""

    def fulfill(self, signed: object, buyer: str) -> int:
        # TODO: gift the buyer the price before the checks run, e.g.:
        #   self.balances[buyer] = self.balances.get(buyer, 0) + int(signed.offer.price)
        return super().fulfill(signed, buyer)


char_clk = FakeClock(STORY_TIME)
charity = CharityChain(char_clk, balances={fx.ADA: SEED, fx.BELL: 0}, next_id=fx.TICKET_ID)
try:
    ticket = charity.fulfill(fx.CANONICAL_SIGNED_OFFER, buyer=MALLORY)
    print("Mallory owns ticket", ticket, "— the funds check was load-bearing")
except InsufficientFunds:
    print("✗ InsufficientFunds — the check still fires (edit the TODO to neuter it)")
# TODO: un-comment once Mallory rides free
# assert charity.owner_of(ticket) == MALLORY and charity.balances[fx.BELL] == int(fx.PRICE_10_TOK)

<details><summary>✅ Solution 12.1 — peek only after trying</summary>

```python
class CharityChain(FakeChain):
    def fulfill(self, signed: object, buyer: str) -> int:
        self.balances[buyer] = self.balances.get(buyer, 0) + int(signed.offer.price)
        return super().fulfill(signed, buyer)
```

Every check in `fulfill` is one `if` line — delete any one and a whole class of theft goes unnoticed, which is why chapter 06 pins each to a named Foundry invariant test.

</details>

## 13. The swap map — every prop's real replacement

The skeleton's promise: swap one prop at a time for the real organ and keep the play
green after every swap. Here is the cast list — each understudy, the role, and the
chapter where the real actor takes the stage:

| cardboard prop | stands in for | the real thing (chapter) |
|---|---|---|
| `FakeClock.advance()` | new blocks moving `block.timestamp` | Anvil's chain time — `06_blockchain_from_zero.ipynb` |
| `FakeChain.fulfill()` + its 4 exceptions | the Settlement contract's checks & reverts | `Settlement.sol` (06) driven via `ChainClient` (07) |
| the `0xabab…` placeholder signature | EIP-712 signing | chainmcp, the only key holder — `07_chainmcp_the_signing_adapter.ipynb` |
| `StubController` (nonce set, session dict) | the controller service | real auth, state machine, HTTP API — `08_controller_the_bouncer.ipynb` |
| `_RESOURCE_MAP` one-entry dict | `resource_map.yaml` (ADR-005) | the controller's resource map (08) |
| `FakeNet` recording calls | gNMI `Set` on SR Linux | `GnmiProvisioner` — `09_netctl_the_hands.ipynb` |
| `ScriptedProvider` / `ScriptedConsumer` | rule 1's two LLM judgment slots | LangGraph agents — `10_agents_the_brains.ipynb` |
| this notebook's hand-wiring | the composition root | worlds + `SKELETON_PROFILE` — `11_worlds_and_profiles.ipynb` |

And one swap has *already happened under your feet* — `FakeNet` is not defined in
`fakes.py` at all anymore. It grew up, moved out to `netctl` (to live beside the real
`GnmiProvisioner` and pass the *same* contract test suite, rule 7), and the skeleton
kept only its stage name:

In [ ]:
from netctl.mock import MockProvisioner

assert FakeNet is MockProvisioner            # a plain alias — the SAME class object
print("FakeNet is netctl.mock.MockProvisioner:", FakeNet is MockProvisioner)
print("FakeNet.__module__ =", FakeNet.__module__, " ← the prop moved out, the play never noticed")
print("✓ the swap machinery works — chapter 09 swaps in the real hands the same way")

## What you learned (and where it goes)

| You did | The concept | Where it goes next |
|---|---|---|
| counted the fake chain: ~80 lines | a fake with behavior is a second implementation that can lie | mock-parity contract tests (09) |
| advanced `FakeClock`; proved `sleep` moves nothing | chain time is the only clock (ADR-004) | Anvil's `evm_increaseTime` (06) |
| ran all four `fulfill` deny paths, asserted **no trace** | checks-before-mutations = hand-made atomicity; the check order is the contract's spec | the real reverts (06/07) |
| poked the scripted agents | rule 1: exactly two judgment slots, here canned | LLM judgment (10) |
| flipped one fact per predicate run; first failure answers | authorization is a pure, ordered checklist | the predicate at home + purity proof (08) |
| challenge → activate → inspect `net.applied` | fetch → decide → act; dependency injection via ports | the real controller & resource map (08), real gNMI (09) |
| tore down by tick and by revocation — then again | two teardown paths funnel into one idempotent teardown (rule 8) | the revocation showpiece, for real (12) |
| sabotaged the salt ledger; the offer settled twice | single-use is a **ledger**, not politeness (invariant I2) | `test_I2_offerSingleUse` in Foundry (06) |

## Experiments to try

Predict the outcome *before* running each one:

- In §5's helper, seed the stage with `balances={fx.ADA: 5 * 10**18, fx.BELL: 0}` (Ada
  has only 5 TOK) and run the happy-path fulfill. Which exception, and does
  `assert_no_trace` still pass?
- In §8, mint a stage where **Bell** is the buyer (`fulfill(..., buyer=fx.BELL)`), then
  have *Ada* try to activate with a fresh nonce. Which `ErrorCode` comes back, and from
  which line of the predicate?
- Change §10's curtain-up time to `fx.WINDOW.end - 1` (one second before close),
  activate, then `tick()` after `advance(1)`. Does the session even reach ACTIVE? Which
  path ends it?
- Deliberate breakage: subclass `StubController` with an `__init__` that calls
  `super().__init__` but then empties `chain._watchers`. Revoke mid-session — what
  state is the session left in, and which real-world failure does that simulate?

## Check yourself

1. Why is `FakeChain` deliberately ~80 lines instead of a faithful mini-Ethereum?
2. A `fulfill` arrives that is expired AND underfunded — which exception is raised, and
   why does that exact answer matter for the real contract in 06?
3. What single property of `fulfill`'s code layout replaces the blockchain's rollback,
   and which helper did we use to *prove* it, four times?
4. Two teardown paths exist. Which rule makes it safe for both to fire on the same
   session, and how did `net.torn_down` make that rule observable?
5. The predicate returned `None` — is that a bug? Whose job was it to turn that `None`
   into an ACTIVE session, and through which port did the provisioning call leave?

**Where this goes next:** `06_blockchain_from_zero.ipynb` — the `FakeChain`'s real
counterpart: accounts, transactions, Anvil, and `Settlement.sol` read section by
section, including the very checks you triggered here.

**Deeper dive:** the compact tour
[`e2e/notebooks/skeleton_explore.ipynb`](../skeleton_explore.ipynb) · the narrative
twin [`docs/03c-skeleton-walkthrough.md`](../../../docs/03c-skeleton-walkthrough.md) ·
this play as CI: [`e2e/tests/test_lifecycle.py`](../../tests/test_lifecycle.py).